# Notebook 3: Flood Susceptibility Mapping

**Flood Susceptibility Mapping — Karabük, Turkey**
CME434, Karabük University

**Purpose:** Apply best model to full Karabük AOI, generate susceptibility raster
**Input:**  `data/raw/Full_AOI_100m.csv`, `outputs/models/best_flood_model.pkl`
**Output:** `outputs/maps/Karabuk_Flood_Susceptibility_100m.tif`,
`outputs/figures/flood_susceptibility_map.png`,
`web/data/flood_susceptibility.geojson`

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import json
import joblib
import os

os.makedirs('../data/output',    exist_ok=True)
os.makedirs('../outputs/maps',   exist_ok=True)
os.makedirs('../outputs/figures',exist_ok=True)
print("Libraries loaded")

## Step 1 — Load Full AOI Dataset

In [ ]:
# TODO: Download Full_AOI_100m.csv from Google Drive folder GIS_Flood_Karabuk
# Place in: data/raw/Full_AOI_100m.csv

full_path = '../data/raw/Full_AOI_100m.csv'
full_df   = pd.read_csv(full_path)
print(f"Full AOI shape: {full_df.shape}")
print(f"First 8 columns: {full_df.columns.tolist()[:8]}")

## Step 2 — Clean Data (same drops as Notebook 1)

In [ ]:
geo_col   = full_df['.geo']
drop_list = ['system:index', '.geo', 'latitude', 'longitude',
             'latitude_1', 'longitude_1']
drop_cols = [c for c in full_df.columns
             if c in drop_list or c.startswith('system')]
features_df = full_df.drop(columns=drop_cols, errors='ignore')
print(f"Feature columns ({features_df.shape[1]}): {features_df.columns.tolist()}")

## Step 3 — Load Best Model and Predict

In [ ]:
model      = joblib.load('../outputs/models/best_flood_model.pkl')
model_type = type(model).__name__
print(f"Model loaded: {model_type}")

if 'SVC' in model_type:
    scaler = joblib.load('../outputs/models/scaler.pkl')
    X_pred = scaler.transform(features_df.values)
    print("Using scaled features (SVM)")
else:
    X_pred = features_df.values
    print("Using raw features (tree-based)")

flood_prob = model.predict_proba(X_pred)[:, 1]
print(f"\nPredictions: {len(flood_prob):,} pixels")
print(f"Range : {flood_prob.min():.3f} — {flood_prob.max():.3f}")
print(f"Mean  : {flood_prob.mean():.3f}")

## Step 4 — Build GeoDataFrame

In [ ]:
def parse_geo(json_str):
    try:
        coords = json.loads(json_str.replace('""', '"'))['coordinates']
        return Point(coords)
    except Exception:
        return None

final_df             = pd.DataFrame({'FloodProb': flood_prob, '.geo': geo_col})
final_df['geometry'] = final_df['.geo'].apply(parse_geo)
final_df             = final_df.dropna(subset=['geometry'])

gdf = gpd.GeoDataFrame(
    final_df[['FloodProb', 'geometry']],
    geometry='geometry', crs='EPSG:4326')
print(f"GeoDataFrame ready: {len(gdf):,} points")

## Step 5 — Classify into 5 Flood Risk Classes

In [ ]:
RISK_COLORS  = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']
RISK_LABELS  = {1: 'Very Low', 2: 'Low', 3: 'Medium', 4: 'High', 5: 'Very High'}

def classify_risk(p):
    if   p < 0.20: return 1
    elif p < 0.40: return 2
    elif p < 0.60: return 3
    elif p < 0.80: return 4
    else:          return 5

gdf['RiskClass'] = gdf['FloodProb'].apply(classify_risk)
gdf['RiskLabel'] = gdf['RiskClass'].map(RISK_LABELS)
gdf['FloodPct']  = (gdf['FloodProb'] * 100).round(1)

print("Risk class distribution:")
print(gdf['RiskLabel'].value_counts().sort_index())
pct_high = (gdf['RiskClass'] >= 4).mean() * 100
print(f"\nHigh + Very High: {pct_high:.1f}% of Karabük")

## Step 6 — Save Shapefile

In [ ]:
shp_path = '../data/output/Karabuk_Flood_Probability.shp'
gdf.to_file(shp_path)
print(f"Saved: {shp_path}  ({len(gdf):,} points, CRS={gdf.crs.to_epsg()})

## Step 7 — Rasterize to GeoTIFF

In [ ]:
gdf_utm    = gdf.to_crs(epsg=32636)
px         = 100
minx, miny, maxx, maxy = gdf_utm.total_bounds
width      = int((maxx - minx) / px) + 1
height     = int((maxy - miny) / px) + 1
transform  = from_origin(minx, maxy, px, px)
meta_base  = dict(driver='GTiff', height=height, width=width,
                  count=1, crs=gdf_utm.crs, transform=transform)

# Probability raster (float)
prob_shapes = ((g, v) for g, v in zip(gdf_utm.geometry, gdf_utm['FloodProb']))
prob_arr    = rasterize(prob_shapes, out_shape=(height, width),
                        transform=transform, fill=np.nan, dtype='float32')
prob_path   = '../outputs/maps/Karabuk_Flood_Susceptibility_100m.tif'
with rasterio.open(prob_path, 'w', **meta_base,
                   dtype='float32', nodata=np.nan) as dst:
    dst.write(prob_arr, 1)
print(f"Saved: {prob_path}")

# Classified raster (int16, 1-5)
cls_shapes  = ((g, v) for g, v in zip(gdf_utm.geometry, gdf_utm['RiskClass']))
cls_arr     = rasterize(cls_shapes, out_shape=(height, width),
                        transform=transform, fill=0, dtype='int16')
cls_path    = '../outputs/maps/Karabuk_Flood_Susceptibility_Classified.tif'
with rasterio.open(cls_path, 'w', **meta_base,
                   dtype='int16', nodata=0) as dst:
    dst.write(cls_arr, 1)
print(f"Saved: {cls_path}")

## Step 8 — Visualize Flood Susceptibility Map

In [ ]:
cmap   = ListedColormap(RISK_COLORS)
bounds = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm   = BoundaryNorm(bounds, cmap.N)

with rasterio.open(cls_path) as src:
    data   = src.read(1).astype(float)
    data[data == 0] = np.nan
    extent = [src.bounds.left, src.bounds.right,
              src.bounds.bottom, src.bounds.top]

fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(data, cmap=cmap, norm=norm, extent=extent)

patches = [mpatches.Patch(color=c, label=f'Class {i+1}: {RISK_LABELS[i+1]}')
           for i, c in enumerate(RISK_COLORS)]
ax.legend(handles=patches, loc='lower right',
          title='Flood Susceptibility', fontsize=9, framealpha=0.9)
ax.set_title('Karabuk Province — Flood Susceptibility Map\n'
             'CME434 | Karabuk University | 2026',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Easting (m, UTM 36N)')
ax.set_ylabel('Northing (m, UTM 36N)')
ax.grid(alpha=0.3, linestyle='--', linewidth=0.5)
plt.tight_layout()

fig_path = '../outputs/figures/flood_susceptibility_map.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## Step 9 — Export GeoJSON for Leaflet Web App

In [ ]:
os.makedirs('../web/data', exist_ok=True)
gdf_web  = gdf.iloc[::3].reset_index(drop=True)  # every 3rd point
web_path = '../web/data/flood_susceptibility.geojson'

gdf_web[['FloodProb', 'RiskClass', 'RiskLabel',
         'FloodPct', 'geometry']].to_file(web_path, driver='GeoJSON')

size_mb = os.path.getsize(web_path) / 1e6
print(f"Saved: {web_path}")
print(f"  Points : {len(gdf_web):,} (every 3rd point)")
print(f"  Size   : ~{size_mb:.1f} MB")
print("\nGeoJSON ready. Open web/index.html in browser.")